# 08 - Baseline cross-validation (RQ1)

Stratified 5-fold cross-validation of the three cheap baselines (lexicon,
Naïve Bayes, logistic regression) on the development pool (`train_fit` +
`validation`, 1620 rows). CV gives a mean ± std F1 per construct, so the
baseline comparison does not rest on a single validation split. The held-out
test set is not touched here. The lexicon rules are imported from
`notebooks/lexicon_rules.py` (the same source notebook 05 uses), so there is no hidden
dependency on any other notebook having been run first.

In [ ]:
# Imports and load

import sys
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Repo root, whether run from notebooks/ or the root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
proc = PROJECT_ROOT / "data" / "processed"

# Import the frozen lexicon from notebooks/ so this notebook stands on its own.
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
from lexicon_rules import predict_all

# Development pool = train_fit + validation (1620). Test stays held out.
pool = pd.read_csv(proc / "train.csv")
labels = ["label_1A", "label_1B", "label_2"]
for c in labels:
    pool[c] = pool[c].astype(int)
print("pool:", pool.shape)

In [ ]:
# Cross-validation

# Shared vectorizer settings (stop_words=None: pronouns/in-group markers are diagnostic).
VEC_KW = dict(lowercase=True, ngram_range=(1, 2), min_df=3, max_df=0.9, stop_words=None)

def cv_scores(pool, labels, n_splits=5, seed=42):
    rows = []
    for label in labels:
        # Stratify per construct so every fold contains positives of THAT label.
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        y = pool[label].values
        lex, nb, lr = [], [], []
        for tr_idx, va_idx in skf.split(pool, y):
            tr, va = pool.iloc[tr_idx], pool.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            # ---- Lexicon: no fitting, just score the held-out fold ----
            key = f"lexicon_{label.replace('label_','')}"
            lex_pred = va["text_clean"].fillna("").apply(lambda t: predict_all(t)[key])
            lex.append(f1_score(y_va, lex_pred, zero_division=0))

            # ---- Naïve Bayes: CountVectorizer, fit on train folds only ----
            cv = CountVectorizer(**VEC_KW)
            Xtr = cv.fit_transform(tr["text_clean"].fillna(""))
            Xva = cv.transform(va["text_clean"].fillna(""))
            nb_m = MultinomialNB(fit_prior=False).fit(Xtr, y_tr)
            nb.append(f1_score(y_va, nb_m.predict(Xva), zero_division=0))

            # ---- Logistic regression: TfidfVectorizer, fit on train folds only ----
            tv = TfidfVectorizer(**VEC_KW)
            Xtr2 = tv.fit_transform(tr["text_clean"].fillna(""))
            Xva2 = tv.transform(va["text_clean"].fillna(""))
            lr_m = LogisticRegression(class_weight="balanced", max_iter=1000).fit(Xtr2, y_tr)
            lr.append(f1_score(y_va, lr_m.predict(Xva2), zero_division=0))

        for name, s in [("lexicon", lex), ("naive_bayes", nb), ("logreg", lr)]:
            rows.append({
                "construct": label.replace("label_", ""),
                "model": name,
                "f1_mean": round(np.mean(s), 3),
                "f1_std":  round(np.std(s), 3),
                "n_pos":   int(y.sum()),
            })
    return pd.DataFrame(rows)

cv_table = cv_scores(pool, labels)
print(cv_table.pivot(index="model", columns="construct", values="f1_mean"))
print("\nfull (mean ± std):")
print(cv_table.to_string(index=False))
cv_table.to_csv(proc / "cv_baseline_metrics.csv", index=False)